In [ ]:
# CELDA 1

"""
Esta celda importa las librerías necesarias, carga el 
dataset y muestra las estadísticas descriptivas completas 
(media, desviación estándar, mínimos, máximos y percentiles)
"""

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

# Cargar el dataset
df = pd.read_csv('dataset_empleados.csv')

# 1. Estadísticas descriptivas completas
print("Estadísticas Descriptivas del Dataset:")
display(df.describe().round(2))

Estadísticas Descriptivas del Dataset:


,edad,anios_en_empresa,salario_mensual,horas_extra_semana,satisfaccion_laboral,num_proyectos_anio,distancia_casa_trabajo_km,ultima_evaluacion_desempeno,capacitaciones_recibidas,tiene_ascenso_ultimos_2_anios,renuncia
count,350.00,350.00,350.00,350.00,350.00,350.00,350.00,350.00,350.00,350.00,350.00
mean,41.78,8.16,1639.01,9.93,3.01,5.39,39.23,0.50,2.51,0.16,0.29
std,11.57,5.89,425.97,5.92,1.45,2.80,22.65,0.29,1.74,0.37,0.45
min,22.00,0.00,460.00,0.00,1.00,1.00,1.00,0.00,0.00,0.00,0.00
25%,32.00,3.00,1339.41,5.00,2.00,3.00,20.00,0.25,1.00,0.00,0.00
50%,43.00,7.00,1605.41,10.00,3.00,5.00,39.00,0.52,3.00,0.00,0.00
75%,52.00,12.00,1951.42,15.00,4.00,8.00,58.75,0.75,4.00,0.00,1.00
max,60.00,20.00,2764.66,20.00,5.00,10.00,80.00,1.00,5.00,1.00,1.00


In [ ]:
# CELDA 2

"""
Aquí usaremos Plotly para generar el mapa de calor. 
Al ser interactivo, podrás pasar el cursor 
por los recuadros para mostrar exactamente los valores 
de correlación con la variable objetivo. 
"""

# 2. Heatmap de correlación
matriz_correlacion = df.corr().round(3)

fig_corr = px.imshow(
    matriz_correlacion,
    text_auto=True, 
    aspect="auto",
    color_continuous_scale='RdBu_r',
    title='Mapa de Calor de Correlaciones'
)

fig_corr.update_layout(width=900, height=800)
fig_corr.show()

In [ ]:
# CELDA 3

"""
Graficar la distribución de la renuncia para evaluar si el 
dataset está balanceado. Usaremos un gráfico de barras interactivo 
que muestre el conteo y el porcentaje.
"""


# 3. Distribución de la variable 'renuncia'
conteo_renuncias = df['renuncia'].value_counts().reset_index()
conteo_renuncias.columns = ['Renuncia (0=No, 1=Sí)', 'Cantidad']

fig_dist = px.bar(
    conteo_renuncias, 
    x='Renuncia (0=No, 1=Sí)', 
    y='Cantidad',
    color='Renuncia (0=No, 1=Sí)',
    text='Cantidad',
    title='Distribución de la Variable Objetivo (Renuncia)',
    color_discrete_sequence=['#2ca02c', '#d62728'] # Verde para 0, Rojo para 1
)

fig_dist.update_traces(textposition='outside')
fig_dist.update_layout(xaxis_type='category')
fig_dist.show()

print("Proporción de clases:")
print(df['renuncia'].value_counts(normalize=True).round(3) * 100)

Proporción de clases:
renuncia
0    71.1
1    28.9
Name: proportion, dtype: float64


In [ ]:
# CELDA 4

"""
Finalmente, necesitamos graficar los boxplots del salario y la satisfacción 
separados por la variable renuncia para identificar patrones visuales.
"""


# 4. Boxplots de Salario y Satisfacción vs Renuncia
# Boxplot de Salario
fig_box_salario = px.box(
    df, 
    x='renuncia', 
    y='salario_mensual', 
    color='renuncia',
    title='Distribución del Salario Mensual según Renuncia',
    labels={'renuncia': 'Renuncia', 'salario_mensual': 'Salario Mensual ($)'},
    color_discrete_sequence=['#2ca02c', '#d62728']
)
fig_box_salario.update_layout(xaxis_type='category')
fig_box_salario.show()

# Boxplot de Satisfacción Laboral
fig_box_satisfaccion = px.box(
    df, 
    x='renuncia', 
    y='satisfaccion_laboral', 
    color='renuncia',
    title='Satisfacción Laboral según Renuncia',
    labels={'renuncia': 'Renuncia', 'satisfaccion_laboral': 'Nivel de Satisfacción (1-5)'},
    color_discrete_sequence=['#2ca02c', '#d62728']
)
fig_box_satisfaccion.update_layout(xaxis_type='category')
fig_box_satisfaccion.show()

In [ ]:
# CELDA 5

"""
debes usar un 80% para entrenamiento y 20% para prueba con un random_state=42 , y debes ajustar el 
StandardScaler solo sobre los datos de entrenamiento para evitar el filtrado de información (data leakage).
"""

# Importar librerías de Scikit-Learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# 5. Dividir el dataset (80% train, 20% test)
X = df.drop('renuncia', axis=1)
y = df['renuncia']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 6. Estandarizar features numéricas
# Identificamos las columnas que son continuas/numéricas puras (excluyendo binarias como el ascenso)
cols_numericas = ['edad', 'anios_en_empresa', 'salario_mensual', 'horas_extra_semana', 
                  'satisfaccion_laboral', 'num_proyectos_anio', 'distancia_casa_trabajo_km', 
                  'ultima_evaluacion_desempeno', 'capacitaciones_recibidas']

scaler = StandardScaler()

# Ajustamos (fit) y transformamos SOLO sobre train
X_train_scaled = X_train.copy()
X_train_scaled[cols_numericas] = scaler.fit_transform(X_train[cols_numericas])

# SOLO transformamos sobre test (usando las reglas aprendidas del train)
X_test_scaled = X_test.copy()
X_test_scaled[cols_numericas] = scaler.transform(X_test[cols_numericas])

print("Tamaño del conjunto de entrenamiento:", X_train_scaled.shape)
print("Tamaño del conjunto de prueba:", X_test_scaled.shape)

Tamaño del conjunto de entrenamiento: (280, 10)
Tamaño del conjunto de prueba: (70, 10)


In [ ]:
# CELDA 6

"""
Ahora vamos a entrenar los tres modelos y a calcular las métricas exigidas: 
Accuracy, Precisión, Recall, F1-Score y AUC-ROC. Crearemos una tabla de Pandas 
para ver los resultados de forma ordenada.
"""

# 7. Inicializar y entrenar los modelos
modelos = {
    'Regresión Logística': LogisticRegression(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

resultados = []

# Bucle para entrenar, predecir y calcular métricas
for nombre, modelo in modelos.items():
    # Entrenar el modelo
    modelo.fit(X_train_scaled, y_train)
    
    # Predecir sobre el conjunto de prueba
    y_pred = modelo.predict(X_test_scaled)
    y_prob = modelo.predict_proba(X_test_scaled)[:, 1] # Probabilidades para AUC-ROC
    
    # 8. Calcular métricas
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    
    # Guardar en la lista
    resultados.append({
        'Modelo': nombre,
        'Accuracy': round(acc, 3),
        'Precisión': round(prec, 3),
        'Recall': round(rec, 3),
        'F1-Score': round(f1, 3),
        'AUC-ROC': round(auc, 3)
    })

# Mostrar la tabla de resultados comparativos
df_resultados = pd.DataFrame(resultados)
print("Métricas de evaluación de los modelos iniciales:")
display(df_resultados)

Métricas de evaluación de los modelos iniciales:


,Modelo,Accuracy,Precisión,Recall,F1-Score,AUC-ROC
0,Regresión Logística,0.743,0.583,0.35,0.438,0.730
1,Random Forest,0.729,0.556,0.25,0.345,0.730
2,Gradient Boosting,0.671,0.412,0.35,0.378,0.696


In [ ]:
# CELDA 7

"""graficar la curva ROC de los 3 modelos en una sola figura para compararlos. Esta curva te ayudará a 
responder visualmente la pregunta sobre por qué un modelo está por encima del otro."""


# 9. Graficar la curva ROC comparativa
import plotly.graph_objects as go
from sklearn.metrics import roc_curve

fig_roc = go.Figure()

# Línea diagonal de referencia (rendimiento aleatorio)
fig_roc.add_shape(type='line', line=dict(dash='dash', color='gray'), x0=0, x1=1, y0=0, y1=1)

for nombre, modelo in modelos.items():
    # Obtener probabilidades de la clase positiva (1 = Renuncia)
    y_prob = modelo.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, umbrales = roc_curve(y_test, y_prob)
    
    # Extraer el AUC calculado previamente en la tabla
    auc_score = df_resultados[df_resultados['Modelo'] == nombre]['AUC-ROC'].values[0]
    
    # Añadir la curva del modelo al gráfico
    fig_roc.add_trace(go.Scatter(
        x=fpr, y=tpr, 
        name=f"{nombre} (AUC = {auc_score})", 
        mode='lines',
        hovertemplate='FPR: %{x:.2f}<br>TPR: %{y:.2f}'
    ))

fig_roc.update_layout(
    title='Curvas ROC Comparativas de los 3 Modelos',
    xaxis_title='Tasa de Falsos Positivos (FPR)',
    yaxis_title='Tasa de Verdaderos Positivos (TPR) / Sensibilidad',
    width=800, height=600,
    hovermode='x unified'
)
fig_roc.show()

In [ ]:
# CELDA 8

"""graficar la matriz de confusión de cada modelo. Estas gráficas te servirán para responder la 
pregunta qué significa cada cuadrante para el negocio"""

# 10. Graficar las matrices de confusión
from sklearn.metrics import confusion_matrix

for nombre, modelo in modelos.items():
    y_pred = modelo.predict(X_test_scaled)
    cm = confusion_matrix(y_test, y_pred)
    
    fig_cm = px.imshow(
        cm, 
        text_auto=True, 
        color_continuous_scale='Blues',
        labels=dict(x="Predicción del Modelo", y="Realidad (Datos Reales)", color="Cantidad"),
        x=['Se Queda (0)', 'Renuncia (1)'],
        y=['Se Queda (0)', 'Renuncia (1)'],
        title=f'Matriz de Confusión: {nombre}'
    )
    
    # Ajustar la posición del texto para mayor claridad
    fig_cm.update_traces(textfont={"size": 20})
    fig_cm.update_layout(width=550, height=550)
    fig_cm.show()

In [ ]:
# CELDA 9

"""Finalmente, para los modelos de ensamble (Random Forest y Gradient Boosting), 
necesitamos graficar qué variables consideraron más importantes para tomar la 
decisión. Esto responde directamente a una de las preguntas de tu informe escrito."""

# 11. Graficar Feature Importance para Random Forest y Gradient Boosting
for nombre in ['Random Forest', 'Gradient Boosting']:
    modelo = modelos[nombre]
    importancias = modelo.feature_importances_
    
    # Crear un DataFrame para ordenar los datos
    df_imp = pd.DataFrame({'Variable': X.columns, 'Importancia': importancias})
    df_imp = df_imp.sort_values(by='Importancia', ascending=True) # Orden ascendente para el gráfico de barras horizontales
    
    fig_imp = px.bar(
        df_imp, 
        x='Importancia', 
        y='Variable', 
        orientation='h',
        title=f'Importancia de las Variables (Feature Importance): {nombre}',
        color='Importancia',
        color_continuous_scale='Viridis'
    )
    
    fig_imp.update_layout(width=850, height=500)
    fig_imp.show()

In [ ]:
# CELDA 10

"""Optimización de Hiperparámetros mediante Validación Cruzada y GridSearchCV"""

from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import pandas as pd

# 12. Aplicar k-fold cross-validation (k=5) al Gradient Boosting base
gb_base = modelos['Gradient Boosting']
cv_scores = cross_val_score(gb_base, X_train_scaled, y_train, cv=5, scoring='f1')

print("--- Validación Cruzada (k=5) ---")
print(f"F1-Score Base: Media = {cv_scores.mean():.3f}, Desviación Estándar = {cv_scores.std():.3f}\n")

# 13. Aplicar GridSearchCV para optimizar 3 hiperparámetros
# Definimos la "rejilla" de parámetros a probar
param_grid = {
    'n_estimators': [50, 100, 200],      # Cantidad de árboles
    'learning_rate': [0.01, 0.05, 0.1],  # Qué tan rápido aprende el modelo
    'max_depth': [3, 4, 5]               # Profundidad máxima de cada árbol
}

# Inicializamos GridSearchCV
grid_search = GridSearchCV(
    estimator=GradientBoostingClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='f1', # Optimizamos para F1-Score debido al desbalance
    n_jobs=-1     # Usa todos los núcleos de tu procesador para ir más rápido
)

print("Ejecutando GridSearchCV... probando 27 combinaciones diferentes (esto tomará unos segundos).")
grid_search.fit(X_train_scaled, y_train)

mejor_gb = grid_search.best_estimator_
print("\n¡Búsqueda terminada! Mejores hiperparámetros encontrados:")
for param, value in grid_search.best_params_.items():
    print(f"- {param}: {value}")

# 14. Comparar el modelo base vs el modelo optimizado
# Predecimos con el nuevo modelo ganador
y_pred_opt = mejor_gb.predict(X_test_scaled)
y_prob_opt = mejor_gb.predict_proba(X_test_scaled)[:, 1]

# Extraemos las métricas de la versión anterior (de la tabla df_resultados)
gb_original_metrics = df_resultados[df_resultados['Modelo'] == 'Gradient Boosting'].iloc[0]

# Creamos la tabla comparativa
comparacion = pd.DataFrame({
    'Métrica': ['Accuracy', 'Precisión', 'Recall', 'F1-Score', 'AUC-ROC'],
    'Gradient Boosting (Base)': [
        gb_original_metrics['Accuracy'], 
        gb_original_metrics['Precisión'], 
        gb_original_metrics['Recall'], 
        gb_original_metrics['F1-Score'], 
        gb_original_metrics['AUC-ROC']
    ],
    'Gradient Boosting (Optimizado)': [
        round(accuracy_score(y_test, y_pred_opt), 3),
        round(precision_score(y_test, y_pred_opt, zero_division=0), 3),
        round(recall_score(y_test, y_pred_opt), 3),
        round(f1_score(y_test, y_pred_opt), 3),
        round(roc_auc_score(y_test, y_prob_opt), 3)
    ]
})

print("\n--- Comparación Final de Métricas ---")
display(comparacion)

--- Validación Cruzada (k=5) ---
F1-Score Base: Media = 0.433, Desviación Estándar = 0.067

Ejecutando GridSearchCV... probando 27 combinaciones diferentes (esto tomará unos segundos).

¡Búsqueda terminada! Mejores hiperparámetros encontrados:
- learning_rate: 0.01
- max_depth: 5
- n_estimators: 200

--- Comparación Final de Métricas ---


,Métrica,Gradient Boosting (Base),Gradient Boosting (Optimizado)
0,Accuracy,0.671,0.729
1,Precisión,0.412,0.533
2,Recall,0.350,0.400
3,F1-Score,0.378,0.457
4,AUC-ROC,0.696,0.720
